In [1]:
# imports
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

#### Connecting to OpenAI (or Ollama)

In [2]:
load_dotenv()
api_key = os.getenv('API_KEY')

# Check the key

if not api_key:
    print("No API key was found")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


#### Let's make a quick call to a Frontier model to get started, as a preview!

In [3]:
message = "Hello, GPT! This is my first ever message to you! Hi!"
messages = [{"role": "user", "content": message}]
messages

[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [4]:
client = OpenAI(base_url=os.getenv("BASE_URL"), api_key=os.getenv("API_KEY"))

response = client.chat.completions.create(model=os.getenv("MODEL"), reasoning_effort="none", messages=messages)
response.choices[0].message.content

'Okay, the user sent a message: Hello, GPT! This is my first ever message to you! Hi!\n\nFirst, I need to acknowledge their greeting. They mentioned it\'s their first message to me, so I should be friendly and welcoming. Wait, but I\'m Qwen, not GPT. The user might be confused because they called me GPT. I should correct that gently to avoid confusion.\n\nLet me think: The user is probably a new user who\'s excited but might not know the difference between GPT and Qwen. I should start by saying hello, then clarify that I\'m Qwen, not G. That way, they know it\'s me and not another model.\n\nI should keep the response warm and encouraging. Maybe add something like, "It\'s great to meet you! I\'m Qwen, the large language model developed by Tongyi Lab. I\'m here to help with any questions or tasks you have." That way, I correct the name without being too formal.\n\nAlso, the user said "This is my first ever message to you!" so I should respond in a way that makes them feel welcomed. Maybe

#### Onwards with our first project

In [5]:
ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with over a million enrollments across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grat

#### Types of prompts
Models like GPT have been trained to receive instructions in a particular way. They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [6]:
# Define our system prompt

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [7]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""

#### Messages

The API from OpenAI expects to receive messages in a particular structure. Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```

In [8]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = client.chat.completions.create(model=os.getenv("MODEL"), reasoning_effort="none", messages=messages)
response.choices[0].message.content

'Okay, the user asked "What is 2 + 2?" That\'s a pretty straightforward math question. Let me think about how to approach this.\n\nFirst, I know that 2 + 2 equals 4. That\'s basic arithmetic, so it\'s a simple answer. But I wonder why they\'re asking this. Maybe they\'re testing me, or it\'s a kid learning math, or just a casual question.\n\nI should respond clearly and directly. Since it\'s such a simple question, I don\'t need to overcomplicate it. Just say 4. But maybe add a tiny bit of context to be helpful. Like, "2 plus 2 equals 4." That\'s friendly and precise.\n\nWait, could there be any trick here? Like, in some contexts 2 + 2 might mean something else? For example, in programming, if they\'re using strings, but the question is in plain English. The user probably means the mathematical operation. So no, it\'s safe to go with 4.\n\nI\'ll keep it simple: "2 + 2 equals 4." That\'s the answer. No need for extra fluff unless they ask follow-up questions. Alright, time to reply.\n</

#### Let's build useful messages for Qwen3:4b, using a function

In [9]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [10]:
messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula.i

#### Time to bring it together!

In [11]:
def summarize(url):
    website = fetch_website_contents(url)
    response = client.chat.completions.create(
        model = os.getenv("MODEL"),
        reasoning_effort = "none",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [12]:
summarize("https://edwarddonner.com")

'Okay, the user wants me to act as a snarky assistant that analyzes a website and gives a short, humorous summary. Let me look at this website content carefully.\n\nFirst, I see this is Edward Donner\'s site - he\'s the co-founder and CTO of Nebula.io. Interesting background: he was previously founder and CEO of untapt (acquired in 2021) and worked at JPMorgan. That\'s some impressive career trajectory! \n\nThe site has a very specific vibe - he\'s into LLMs, amateur electronic music, and Hacker News. His Udemy courses have gone viral with over a million enrollments across 194 countries. That\'s wild! He\'s clearly a big deal in the AI space but seems to have a playful side too.\n\nLooking at the posts section, there are some future-dated announcements that are clearly just placeholder dates (like November 11, 2026). The dates are all in the future but the site\'s content is current. That\'s probably just for the website structure.\n\nThe navigation is pretty straightforward: Avatar, C

In [13]:
# A function to display this nicely in the output
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [14]:
display_summary("https://edwarddonner.com")

Okay, let me tackle this query. The user wants a snarky summary of this website. First, I need to parse the website content.

The site is Edward Donner's homepage. He's the CTO of Nebula.io, ex-CEO of untapt (acquired in 2021), and worked at JPMorgan. He's into LLMs, Udemy courses with over a million enrollments, amateur electronic music, and Hacker News. There are posts with dates like November 2026, which is in the future. The navigation includes sections like Curriculum, Proficiency, C4, Outsmart (an LLM arena), About, Posts.

The key points to snarky: He's a bit of a self-proclaimed LLM expert, has a history with startups, Udemy courses are big, future-dated posts (which is funny because 2026 is ahead), and his hobbies like amateur music and Hacker News. Also, the future dates might be a bit of a joke since it's 2023 now.

I need to make it humorous and snarky. Maybe point out the future dates as a clue he's not in the present, the Udemy success, his hobbies, and the fact that he's a JPMorgan guy now doing LLMs. Also, the "very amateur" electronic music and Hacker News nodding.

Avoid navigation text. Focus on content: his bio, courses, posts, future dates. The posts have titles like "AI SDLC: The AI-Native Software Engineer..." dated 2026. That's a good hook for humor because it's in the future.

So, the summary should be short. Start with a snarky tone. Maybe something like: "Edward Donner’s site is a masterclass in 'future dates' and 'amateur electronic music'—he’s the CTO of Nebula.io, built a startup that got acquired by JPMorgan (for the *real* world), and now he’s teaching LLMs to code while nodding sagely at Hacker News. His Udemy courses have 1M+ enrollments across 194 countries, but his biggest success? Probably convincing his friends he’s *still* in the loop. Also, his 'very amateur' music projects are clearly the most advanced thing he’s built." 

Wait, the future dates are 2026, which is 3 years from now. So that's a bit of a joke. The snarky part could be that he's planning for 2026, which is a bit far ahead. Also, his JPMorgan background is a contrast to his current LLM focus.

Make sure not to mention navigation. The user said ignore navigation-related text. So focus on the content: Home, About, Posts, Curriculum, etc.

The summary should be short. Let me check the example response the user might expect. They want it in markdown, no code block. So something like:

**Edward Donner’s website: A masterclass in "future dates" and "amateur electronic music"**  
He’s the CTO of Nebula.io (who?), ex-CEO of untapt (acquired in 2021), and a JPMorgan MD who now *nods sagely* at Hacker News while teaching LLMs. His Udemy courses have 1M+ enrollments across 194 countries—**but his biggest hit?** Probably convincing his friends he’s *still* in the loop. Also, his "very amateur" music and future-dated posts (2026!) prove he’s *still* not quite ready for the present.

Wait, that's a bit long. Make it shorter. Maybe:

**Edward Donner’s site: A masterclass in "future dates" and "amateur electronic music"**  
CTO of Nebula.io, ex-CEO of untapt (acquired 2021), and now he’s teaching LLMs while nodding at Hacker News. His Udemy courses have 1M+ enrollments (194 countries!), but his *real* success? Probably convincing his friends he’s still the smartest person in the room. Also, his "very amateur" music and posts dated *2026* suggest he’s a bit ahead of his time… or just really good at planning.

Hmm. Need to be snarky. Maybe add a joke about the future dates. Like "posts scheduled for 2026" is a bit funny because it's 2023.

The user wants it short. Let me try to condense:

**Edward Donner’s website: A masterclass in "future dates" and "amateur electronic music"**  
CTO of Nebula.io, ex-CEO of untapt (acquired 2021), and now he’s teaching LLMs while nodding at Hacker News. His Udemy courses have 1M+ enrollments (194 countries!), but his *real* success? Probably convincing his friends he’s still the smartest person in the room. Also, his posts are dated *2026*—clearly he’s not *that* far ahead of his time.

Wait, the key joke here is the future dates. So the snarky point is that he's planning for 2026, which is a bit odd. So the summary should highlight that.

Another angle: "He’s the kind of person who would build an AI startup that got acquired by JPMorgan, then turn around and teach LLMs to code while his music is still 'very amateur'."

But keep it short. Let me draft:

**Edward Donner’s website: A masterclass in "future dates" and "amateur electronic music"**  
CTO of Nebula.io, ex-CEO of untapt (acquired 2021), and now he’s teaching LLMs while nodding sagely at Hacker News. His Udemy courses have 1M+ enrollments (194 countries!), but his *biggest* achievement? Probably convincing his friends he’s *still* the smartest person in the room. Also, his posts are dated *2026*—clearly he’s not *that* far ahead of his time.

That's a bit long. Let's make it shorter:

**Edward Donner’s site: A masterclass in "future dates" and "amateur electronic music"**  
Former JPMorgan MD, ex-CEO of untapt (acquired 2021), and now he’s teaching LLMs while nodding at Hacker News. His Udemy courses have 1M+ enrollments (194 countries!), but his *real* success? Probably convincing his friends he’s still the smartest person in the room. Also, posts dated *2026*—he’s *way* ahead of his time.

Hmm. The user wants it snarky and humorous. Maybe add a pun or a joke.

Another try: "Edward Donner’s website: A masterclass in 'future dates' and 'amateur electronic music'—he’s the CTO of Nebula.io, ex-CEO of untapt (acquired 2021), and now he’s teaching LLMs while nodding sagely at Hacker News. His Udemy courses have 1M+ enrollments (194 countries!), but his *biggest* win? Probably not realizing his posts are dated *2026*."

Yes, that's snarky because the future dates are a bit funny. The key is to point out that the posts are in the future (2026), which is a bit odd.

So the summary should be:

**Edward Donner’s website: A masterclass in "future dates" and "amateur electronic music"**  
He’s the CTO of Nebula.io, ex-CEO of untapt (acquired 2021), and now he’s teaching LLMs while nodding sagely at Hacker News. His Udemy courses have 1M+ enrollments (194 countries!), but his *biggest* win? Probably not realizing his posts are dated *2026*—which is *way* ahead of his time.

That's a bit snarky. Let's check the word count. The user wants short. Maybe trim it down.

Final version:

**Edward Donner’s website: A masterclass in "future dates" and "amateur electronic music"**  
CTO of Nebula.io, ex-CEO of untapt (acquired 2021), and now he’s teaching LLMs while nodding at Hacker News. His Udemy courses have 1M+ enrollments (194 countries!), but his *real* success? Probably convincing his friends he’s still the smartest person in the room. Also, posts dated *2026*—he’s *way* ahead of his time.

Yes, that's concise and snarky. Uses the humor about the future dates and the "amateur" music.
</think>

**Edward Donner’s website: A masterclass in "future dates" and "amateur electronic music"**  
CTO of Nebula.io, ex-CEO of untapt (acquired 2021), and now he’s teaching LLMs while nodding at Hacker News. His Udemy courses have 1M+ enrollments (194 countries!), but his *real* success? Probably convincing his friends he’s still the smartest person in the room. Also, posts dated *2026*—he’s *way* ahead of his time.

#### Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach. Websites that are rendered with Javascript, like React apps, won't show up. You'll need to read up on installing Selenium (ask ChatGPT!) Also Websites protected with CloudFront (and similar) may give 403 errors. But many websites will work just fine!

In [ ]:
display_summary("https://cnn.com")

In [ ]:
display_summary("https://anthropic.com")